In [1]:
from pathlib import Path
import pandas as pd
import yaml
import numpy as np

def get_project_root():
    return Path().resolve().parent

policy_encoding = pd.read_csv(get_project_root() / "data" / "raw" / "policy_encoding.csv")

path = get_project_root() / "data" / "mapping" / "index_schema.yaml"
with path.open("r", encoding="utf-8") as f:
    index_schema = yaml.safe_load(f)

def build_rename_map(schema: dict) -> dict:
    rename_map = {}

    for domain, items in schema.items():
        for key, spec in items.items():
            if isinstance(spec, dict) and "column" in spec and "indicator" in spec:
                rename_map[spec["column"]] = key

    return rename_map

# rename
rename_map = build_rename_map(index_schema)
rename_map['City/Regency'] = 'regency'
rename_map['POLICY ADOPTION'] = 'policy_version'
rename_map['Number of reported VAW cases (SIMFONI)'] = 'vaw_reported_cases'
rename_map['Gross Regional Domestic Product (GDRP) at current market prices (billion Rupiah)']='gdp'
rename_map["Adoption of regulation establishing an Integrated Service Unit for Women and Children's Protection"]='adoption_isu'
rename_map['Year'] = 'year'

policy_encoding = policy_encoding.drop(
    columns=["New city/regency year of legal establishment"]
)
policy_encoding = policy_encoding.rename(columns=rename_map)

# immediately remove domain count columns (need to recalculate, some manual errors and updates to come)
# get rid of those all-caps columns (domains)
mask = ~policy_encoding.columns.str.match(r"^[A-Z0-9\s/&\-()]+$")
policy_encoding = policy_encoding.loc[:, mask]

# map this to fullname (master)
path = get_project_root() / "data" / "mapping" / "city_mapping.yaml"
with path.open("r", encoding="utf-8") as f:
    city_mapping = yaml.safe_load(f)
city_mapping_reversed = {v: k for k, v in city_mapping.items()}

# rename and reorder columns
policy_encoding['fullname'] = policy_encoding['regency'].map(city_mapping_reversed)
policy_encoding = policy_encoding[["fullname"] + [c for c in policy_encoding.columns if c != "fullname"]]
policy_encoding = policy_encoding.drop(columns=["regency", "Province"],)

# add a column for new policy adoption (i.e. a new policy or a policy update)
policy_encoding = policy_encoding.sort_values(["fullname", "year"])
policy_encoding["policy_change"] = (
    policy_encoding.groupby("fullname")["policy_version"]
    .diff()
    .isin([1])
    .astype(int)
)
policy_encoding["has_policy"] = (policy_encoding["policy_version"] != 0).astype(int)

# add filename variable
policy_encoding["filename"] = np.select(
    [
        policy_encoding["policy_version"].eq(0),
        policy_encoding["policy_version"].eq(1),
        policy_encoding["policy_version"].eq(2),
    ],
    [
        "",
        policy_encoding["fullname"].str.replace(" ", "_") + ".txt",
        policy_encoding["fullname"].str.replace(" ", "_") + "_v2.txt",
    ],
    default=""
)

##### update raw data here ######
corr_path = (get_project_root()/ "data"/ "corrections"/ "manual_overwrites.yaml")

with corr_path.open("r", encoding="utf-8") as f:
    corrections = yaml.safe_load(f) or {}

for filename, fixes in corrections.items():
    mask = policy_encoding["filename"].eq(filename) & policy_encoding["policy_change"]==1

    for col, value in fixes.items():
        policy_encoding.loc[mask, col] = value


# read in master mapping and map code
master = pd.read_csv(get_project_root() / "data" / "mapping" / "master_2024.csv")
policy_encoding = policy_encoding.merge(
    master[['fullname', 'value_kab']],
    on='fullname',
    how='left'
)
policy_encoding['ADM2_PCODE'] = 'ID' + policy_encoding['value_kab'].astype(str)

# save processed policy_encoding.csv
policy_encoding = policy_encoding.reset_index(drop=True)
out_path = get_project_root() / "data" / "processed" / "policy_encoding.csv"
policy_encoding.to_csv(out_path, index=False)


## Compute sums and Index

In [2]:
# Index: save wide format for validation

# calculate count columns and index data
for domain, indicators in index_schema.items():
    cols = list(indicators.keys())
    # as a count
    #policy_encoding[domain] = policy_encoding[cols].sum(axis=1)
    # as a percentage
    policy_encoding[domain] = (
        policy_encoding[cols].sum(axis=1, skipna=False)
        / len(cols)
    ) * 100

coverage = [
    'scope_of_violence',
    'primary_prevention',
    'intersectional_approach',
]

implementation = [
    'institutional_mechanism',
    'specialised_support_services',
    'stakeholder_engagement',
    'budget_funding_sources',
]

policy_encoding = policy_encoding.assign(
    coverage_subindex=policy_encoding[coverage].mean(axis=1, skipna=False),
    implementation_subindex=policy_encoding[implementation].mean(axis=1, skipna=False),
)

policy_encoding['quality_index'] = policy_encoding[
    ['coverage_subindex', 'implementation_subindex']
].mean(axis=1, skipna=False)

# forward fill relevant columns
cols_to_ffill = [c for c in policy_encoding.columns
        if c not in ['adoption_isu', 'vaw_reported_cases', 'gdp', 'policy_change']]
policy_encoding[cols_to_ffill] = (
    policy_encoding
    .groupby('fullname')[cols_to_ffill]
    .ffill()
)

# save as procecssed wide (for valudation against GDP, reporting data)
out_path = get_project_root() / "data" / "processed" / "wide_validation_index.csv"
policy_encoding.to_csv(out_path, index=False)


## AI model data

In [3]:
policy_encoding = policy_encoding[policy_encoding['policy_change']==1]

indicator_ids = [
    indicator_id
    for domain in index_schema.values()
    for indicator_id in domain.keys()
]

# Wide -> long
cols = ['fullname', 'year', 'filename'] + indicator_ids

policy_long = policy_encoding[cols].melt(
    id_vars=['fullname', 'filename', 'year'],
    value_vars=indicator_ids,
    var_name='indicator_id',
    value_name='value'
)

# resort (so 1.1 is not followed by 1.10)
policy_long["indicator_id"] = pd.Categorical(
    policy_long["indicator_id"],
    categories=indicator_ids,
    ordered=True
)

policy_long = (
    policy_long
    .sort_values(["fullname", "year", "indicator_id"])
    .reset_index(drop=True)
)

indicator_lookup = {
    indicator_id: details["indicator"]
    for domain in index_schema.values()
    for indicator_id, details in domain.items()
}

policy_long["indicator_value"] = (
    policy_long["indicator_id"]
    .map(indicator_lookup)
)

out_path = get_project_root() / "data" / "processed" / "long_policy_encoding.csv"
policy_long.to_csv(out_path, index=False)

In [4]:
policy_long

,fullname,filename,year,indicator_id,value,indicator_value
0,ACEH BIREUEN,ACEH_BIREUEN.txt,2022,1.1,1.0,Domestic violence
1,ACEH BIREUEN,ACEH_BIREUEN.txt,2022,1.2,1.0,Sexual violence
2,ACEH BIREUEN,ACEH_BIREUEN.txt,2022,1.3,1.0,Physical violence
3,ACEH BIREUEN,ACEH_BIREUEN.txt,2022,1.4,1.0,Psychological violence
4,ACEH BIREUEN,ACEH_BIREUEN.txt,2022,1.5,1.0,Economic violence
...,...,...,...,...,...,...
11083,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,2017,7.1,1.0,Local government budget
11084,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,2017,7.2,0.0,National government budget
11085,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,2017,7.3,0.0,International donors
11086,SUMATERA UTARA PAKPAK BHARAT,SUMATERA_UTARA_PAKPAK_BHARAT.txt,2017,7.4,1.0,Other legal sources


In [5]:
#policy_encoding[policy_encoding["fullname"] == "NUSA TENGGARA TIMUR TIMOR TENGAH UTARA"]

In [8]:
policy_encoding[policy_encoding["filename"] == "JAWA_BARAT_BANDUNG_KOTA.txt"]['2.3']

1671    1.0
Name: 2.3, dtype: float64